Т.к мы ищем датасет или бенчмарк, который будет сильно зависеть от retrieval'a, то есть будем придерживаться простого правила, если при замене корпуса в котором ищем факт ответ не изменится, то он нам не подходит, потому что это уже будет проврека general  LLM knowledge и никак не зависит от нашего датасета.

Также стоит обращать свнимание на содержание датасета, мы не может обучать Policy LLM просто на QA без контекста поэтому будем ориентироваться на датасет где есть Q + Evidence/support pessages + A

# FEVER

In [14]:
import json

Для начала был рассмотрен датасет FEVER, данные вряд ли подойдут для последующего GRPO потому что ответы зачастую короткие и на этом вряд ли получится сформировать атомарные гипотезы $(c_1, ... , c_n)$, так же не является классическим вопросно-ответным датасетом (QA), что в последующем сильно усложнит реализацию

In [15]:
DATA_PATH = "../data/FEVER/paper_dev.jsonl" 

In [16]:
dataset = [json.loads(sample) for sample in open(DATA_PATH)]

In [17]:
len(dataset)


9999

In [18]:
dataset[0]

{'id': 91198,
 'verifiable': 'NOT VERIFIABLE',
 'label': 'NOT ENOUGH INFO',
 'claim': 'Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League.',
 'evidence': [[[108548, None, None, None]]]}

# ASQA

In [5]:
import pandas as pd

In [6]:
data = pd.read_parquet("../data/asqa/dev-00000-of-00001-58a9a40c6e69f07b.parquet", engine="pyarrow")

In [7]:
data.head()

,ambiguous_question,qa_pairs,wikipages,annotations,sample_id
0,Who has the highest goals in world football?,"[{'context': 'No context provided', 'question'...",[{'title': 'International Federation of Footba...,"[{'knowledge': [], 'long_answer': 'Ali Dael ha...",-7013890438520559398
1,Who is the original artist of sound of silence?,[{'context': 'Sounds of Silence is the second ...,"[{'title': 'The Sound of Silence', 'url': 'htt...",[{'knowledge': [{'content': 'Wednesday Morning...,7089015503030534342
2,When was the first apple i phone made?,"[{'context': 'On January 9, 2007, Steve Jobs a...","[{'title': 'iPhone (1st generation)', 'url': '...",[{'knowledge': [{'content': 'The iPhone was re...,8793099883447006698
3,Who played the weasley brothers in harry potter?,[{'context': 'Richard Fish appeared as Bill br...,"[{'title': 'List of Harry Potter characters', ...",[{'knowledge': [{'content': 'Dozens of actors ...,-881464876144297194
4,How many state parks are there in virginia?,"[{'context': 'No context provided', 'question'...","[{'title': 'List of Virginia state parks', 'ur...",[{'knowledge': [{'content': 'Virginia opened i...,1650309494326541834


In [8]:
data.iloc[0]["ambiguous_question"]

'Who has the highest goals in world football?'

In [9]:
data.iloc[0]["qa_pairs"]

array([{'context': 'No context provided', 'question': "Who has the highest goals in men's world international football?", 'short_answers': array(['Daei', 'Ali Daei'], dtype=object), 'wikipage': None},
       {'context': 'No context provided', 'question': "Who has the highest goals all-time in men's football?", 'short_answers': array(['Bican', 'Josef Bican'], dtype=object), 'wikipage': None},
       {'context': 'The first player to reach 100 international goals was Italian Elisabetta Vignotto. Abby Wambach scored 100 goals in 9 years, while Christine Sinclair reached the milestone in just under 10 years while Mia Hamm is the youngest player to score 100 international goals at the age of 26 years 185 days. Most played exclusively in the forward position, with Kristine Lilly and Michelle Akers having also played as midfielder. All players scored at a high average rate of more than one goal every three matches. International goals in this list should not include goals scored in penalty-sho

Слабо подходит общие факты и не очень длинные ответы + мало контекста или его вообще нет

# MISUQUE

In [10]:
DATA_PATH = "../data/misuque/musique_ans_v1.0_dev.jsonl" 

In [11]:
dataset = [json.loads(sample) for sample in open(DATA_PATH)]

In [12]:
dataset[0]["question"]

'Who is the spouse of the Green performer?'

In [14]:
subsamples = []

In [15]:
for sample in dataset:
    if len(sample["question"]) >= 100:
        subsamples.append(sample)

In [17]:
len(subsamples)

1079

In [21]:
subsamples[0].keys()

dict_keys(['id', 'paragraphs', 'question', 'question_decomposition', 'answer', 'answer_aliases', 'answerable'])

In [25]:
subsamples[0]["answer"]

'Mount Laurel Township'

In [23]:
subsamples[0]["question"]

'What is the place of birth of the person who submitted the first version of the equal rights amendment to congress in 1923?'

In [24]:
subsamples[0]["paragraphs"][:5]

[{'idx': 0,
  'title': 'Equal Rights Amendment',
  'paragraph_text': "Alice Paul, the head of the National Women's Party, believed that the Nineteenth Amendment would not be enough to ensure that men and women were treated equally regardless of sex. In 1923, she revised the proposed amendment to read:",
  'is_supporting': True},
 {'idx': 1,
  'title': 'Twenty-second Amendment to the United States Constitution',
  'paragraph_text': 'The Twenty - second Amendment (Amendment XXII) to the United States Constitution sets a limit on the number of times a person is eligible for election to the office of President of the United States, and also sets additional eligibility conditions for presidents who succeed to the unexpired terms of their predecessors. Congress approved the amendment on March 24, 1947, and submitted it to the state legislatures for ratification. That process was completed on February 27, 1951, after the amendment had been ratified by the requisite 36 of the then - 48 states 

In [27]:
subsamples[0]

{'id': '2hop__61318_362941',
 'paragraphs': [{'idx': 0,
   'title': 'Equal Rights Amendment',
   'paragraph_text': "Alice Paul, the head of the National Women's Party, believed that the Nineteenth Amendment would not be enough to ensure that men and women were treated equally regardless of sex. In 1923, she revised the proposed amendment to read:",
   'is_supporting': True},
  {'idx': 1,
   'title': 'Twenty-second Amendment to the United States Constitution',
   'paragraph_text': 'The Twenty - second Amendment (Amendment XXII) to the United States Constitution sets a limit on the number of times a person is eligible for election to the office of President of the United States, and also sets additional eligibility conditions for presidents who succeed to the unexpired terms of their predecessors. Congress approved the amendment on March 24, 1947, and submitted it to the state legislatures for ratification. That process was completed on February 27, 1951, after the amendment had been rat

Выводы по данному датасету:

[+] Нельзя ответить по 1 фатку  \
[+] Есть доказательства \
[+] Подходит для retrieval'a \
[x] Не подойдет для Falsification 

# QASPER

In [1]:
import pandas as pd

In [2]:
dataset = pd.read_parquet("../data/Qasper/0000.parquet")

In [4]:
dataset.head()

,id,title,abstract,full_text,qas,figures_and_tables
0,1911.10742,End-to-End Trainable Non-Collaborative Dialog ...,End-to-end task-oriented dialog models have ac...,"{'section_name': ['Introduction', 'Related Wor...",{'question': ['How big is the ANTISCAM dataset...,{'caption': ['Table 1: Hierarchical intent ann...
1,1904.09131,OpenTapioca: Lightweight Entity Linking for Wi...,We propose a simple Named Entity Linking syste...,"{'section_name': ['Introduction', 'Particulari...",{'question': ['What is the accuracy of this mo...,{'caption': ['Figure 1: Example of an annotate...
2,1611.06322,Spotting Rumors via Novelty Detection,Rumour detection is hard because the most accu...,"{'section_name': ['Introduction', 'Related Wor...",{'question': ['What previous methods do they c...,{'caption': ['Table 1: Excerpt of topics with ...
3,1604.02038,Sentence Level Recurrent Topic Model: Letting ...,We propose Sentence Level Recurrent Topic Mode...,"{'section_name': ['Introduction', 'Related Wor...",{'question': ['What baselines did they compare...,{'caption': ['Figure 1: The illustration of th...
4,1911.04474,TENER: Adapting Transformer Encoder for Named ...,The Bidirectional long short-term memory netwo...,"{'section_name': ['Introduction', 'Related Wor...",{'question': ['Which NER dataset do they use?'...,{'caption': ['Figure 1: An example for NER. Th...


In [10]:
dataset.iloc[0]["qas"]["question"]

array(['How big is the ANTISCAM dataset? ', 'How is intent annotated?',
       'What are the baselines outperformed by this work?',
       'What are the evaluation metrics and criteria used to evaluate the model performance?'],
      dtype=object)

In [13]:
dataset.iloc[0]["full_text"]

{'section_name': array(['Introduction', 'Related Work',
        'Non-Collaborative Task Annotation Scheme', 'Datasets',
        'Datasets ::: AntiScam Dataset',
        'Datasets ::: PersuasionForGood Dataset', 'Model ::: Background',
        'Model ::: Intent and Semantic Slot Classifiers',
        'Model ::: Response Generation', 'Model ::: Response Filtering',
        'Experiments', 'Experiments ::: Baseline Models',
        'Experiments ::: Automatic Evaluation Metrics',
        'Experiments ::: Human Evaluation Metrics', 'Results and Analysis',
        'Conclusion and Future Work', 'Acknowledgements',
        'Appendix ::: Anti-Scam Collection Setting',
        'Appendix ::: Training details', 'Appendix ::: Example Dialog'],
       dtype=object),
 'paragraphs': array([array(['Considerable progress has been made building end-to-end dialog systems for collaborative tasks in which users cooperate with the system to achieve a common goal. Examples of collaborative tasks include making

Что-то уже подходящее, подходит под RAG можно искаль false evidence и ответы/вопросы однозначно сильно полагаются на данные статей в датасете